# Sabotage Connectome

Compares two connectome-based source reconstruction solvers -- `wCSET` (`harmonic_method="lowpass"`, `harm=5`, the value settled on in `harmonic-prior-tuning.ipynb`) and `CSET` (per-timepoint iterative L1 solve, `lambda2=0.1`) -- against 3 independent connectome corruption methods (`swap`, `brutal`, `dmri`; see the "Three corruption methods" section) at increasing severity, evaluated against `simf10`'s known ground truth (`gold`), now across **multiple simulated runs** rather than a single fixed one.

Reconstruction is restricted to just the half-rising-phase and peak-phase timepoints (`hrp`/`pp`): both solvers process each timepoint fully independently (no temporal coupling -- see their `solve()` implementations), so this is a real cost reduction, not an approximation, and it matters a lot for `CSET` specifically, whose per-timepoint iterative L1 optimization is far more expensive than `wCSET`'s single linear solve.

**Two-stage parallel design** (see chat history for the full design discussion):
- **Stage 1** (`mm.analysis.connectivity.cache_cst_batch`): corrupt + eigendecompose every `(method, percentage, repeat)` connectome *in parallel*, each inside its own worker process, writing straight to disk -- never holding more than `n_jobs` connectome/eigenvector pairs (~450MB each) in memory at once, regardless of total sweep size.
- **Stage 2** (`mm.analysis.functional.inverse.run_parallel_pipeline`, unmodified): sequentially loop over the now-cached connectomes, and for each, parallelize reconstruction+scoring *across runs* -- exactly the same pipeline `connectome_surrogates.ipynb` already uses, just fed one connectome at a time via closure instead of a fixed dict of named surrogates.

**Small first pass**: `methods` = all 3, `percentages=[0, 100]`, `n_repeats=1`, `runs=["1","2","3"]` -- proves the two-stage design end-to-end (correctness already verified separately at even smaller scale) before committing to the full sweep (finer percentage grid, more repeats, more runs). Another parallel job is running on this machine, so Stage 1 is deliberately thread-capped rather than tuned for raw speed -- see its call below.

=== EXECUTED === (filled in after running)

## 🧪 - Experiment setup
- `fwd`: Forward solution
- `con`/`con_strength`: Structural connectome (raw, then length -> strength)
- `source_space`: `fwd["src"]`, used for `dmri`'s distance-biased corruption weights
- `eigvec_clean`/`eigval_clean`: Precomputed eigendecomposition of the clean connectome (reused at `percentage=0`, where every corruption method is a no-op, for every method's curve)
- `dipole_positions`: for peak-localization-error scoring
- `runs`: which `simf10` simulated runs to evaluate against (unlike the single-run version of this notebook, `ave`/`ncov`/`gold`/`source` are no longer loaded here -- they're per-run and loaded inside `run_parallel_pipeline`'s workers via `InverseSimTask` paths)

In [1]:
# imports
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

import mindmap as mm
import mne

from mindmap.analysis.functional.common.metric import find_peak_localization_error
from mindmap.analysis.simulation.engine import Simulator
from mindmap.analysis.connectivity import ConnectomeCacheTask, cache_cst_batch
from mindmap.analysis.functional.inverse import InverseSimTask, run_parallel_pipeline

# readers
dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"

# subject-level data (run-independent)
fwd = dm.load(sub, "fwd", desc="inv")
con = dm.load(sub, "con")
eig = dm.load(sub, "eig")
eigvec_clean, eigval_clean = eig["vec"], eig["val"]

con_strength = np.divide(1, con, out=np.zeros_like(con), where=con > 0)
source_space = fwd["src"]
dipole_positions = mm.anatomy.source_space.get_dipole_positions(fwd["src"]) * 1000
simulator_dir = dm.roots["sim"]/f"sub-{sub}"/"simulator"

## 🎚️ - Three corruption methods, calibrated to a common severity axis

Unchanged from the earlier version of this notebook (see chat history for the full
reasoning): `swap` (`swap_rewire_connectome`, degree/edge-count-preserving, needs
empirical calibration since its swap-count -> corruption curve saturates below 100%
and isn't linear), `brutal` (`perturb_connectome_topology`, uniform sampling, exact
closed-form severity), and `dmri` (`perturb_connectome_topology` again, now with
distance-biased sampling weights mimicking real tractography error structure).

In [2]:
percentages = [0, 20, 40, 60, 80, 100]
n_repeats = 5
methods = ["swap", "brutal", "dmri"]
DMRI_ADDITION_RATIO = 2   # additions:deletions -- see markdown above

triu_idx = np.triu_indices_from(con_strength, k=1)
orig_vals = con_strength[triu_idx]
orig_edges = orig_vals != 0
total = int(orig_edges.sum())

# --- swap: probe a swap-count grid, invert the curve to hit pct/100 * ceiling.
def measure_swap_binary_frac(mult, seed=0):
    n_swaps = round(mult * total / 2)
    rewired = mm.analysis.connectivity.swap_rewire_connectome(con_strength, n_swaps=n_swaps, seed=seed)
    return ((rewired[triu_idx] != 0) & ~orig_edges).sum() / total

mult_grid = np.array([0.05, 0.1, 0.2, 0.35, 0.5, 0.7, 1, 1.5, 2, 3, 4, 5, 7, 10, 13, 16, 20, 25, 30])
probe_fracs = np.array([measure_swap_binary_frac(mult) for mult in mult_grid])
probe_fracs_mono = np.maximum.accumulate(probe_fracs)  # enforce monotonic, needed for interpolation
swap_ceiling = probe_fracs_mono[-1]

calibrated_n_swaps = {0: 0}
for percentage in percentages[1:]:
    target_frac = percentage / 100 * swap_ceiling
    mult = np.interp(target_frac, probe_fracs_mono, mult_grid)
    calibrated_n_swaps[percentage] = round(mult * total / 2)

# --- brutal / dmri: exact closed form, no probing needed (see markdown above).
brutal_counts = {p: round(p / 100 * total) for p in percentages}
dmri_counts = {p: (round(brutal_counts[p] / DMRI_ADDITION_RATIO), brutal_counts[p]) for p in percentages}  # (n_deletions, n_additions)

deletion_weights, addition_weights = mm.analysis.connectivity.dmri_uncertainty_weights_from_source_space(con_strength, source_space)

print(f"swap_rewire_connectome empirical ceiling (treated as '100%'): {swap_ceiling:.4f}")
print(f"calibrated n_swaps per severity level: {calibrated_n_swaps}")
print(f"brutal n_deletions=n_additions per severity level: {brutal_counts}")
print(f"dmri (n_deletions, n_additions) per severity level: {dmri_counts}")


def corrupt(method, percentage, seed):
    if method == "swap":
        return mm.analysis.connectivity.swap_rewire_connectome(con_strength, n_swaps=calibrated_n_swaps[percentage], seed=seed)
    if method == "brutal":
        k = brutal_counts[percentage]
        return mm.analysis.connectivity.perturb_connectome_topology(con_strength, n_deletions=k, n_additions=k, seed=seed)
    if method == "dmri":
        n_del, n_add = dmri_counts[percentage]
        return mm.analysis.connectivity.perturb_connectome_topology(
            con_strength, n_deletions=n_del, n_additions=n_add,
            deletion_weights=deletion_weights, addition_weights=addition_weights, seed=seed,
        )
    raise ValueError(f"unknown method {method!r}")

swap_rewire_connectome empirical ceiling (treated as '100%'): 0.9890
calibrated n_swaps per severity level: {0: 0, 20: 24335, 40: 55744, 60: 101659, 80: 182560, 100: 3274785}
brutal n_deletions=n_additions per severity level: {0: 0, 20: 43664, 40: 87328, 60: 130991, 80: 174655, 100: 218319}
dmri (n_deletions, n_additions) per severity level: {0: (0, 0), 20: (21832, 43664), 40: (43664, 87328), 60: (65496, 130991), 80: (87328, 174655), 100: (109160, 218319)}


## 🧵 - Stage 1: parallel corrupt + eigendecompose + cache

One `ConnectomeCacheTask` per `(method, percentage, repeat)` with `percentage > 0`
(`percentage=0` is a no-op for every method, so it just reuses `eigvec_clean`/
`eigval_clean` directly in Stage 2 below -- no need to cache it). Each task's
connectome is corrupted, eigendecomposed, and written to `cache_dir/{label}.npz`
*inside its own worker process* -- the connectome and its eigenvectors never
travel back to this notebook's process.

`n_jobs=2, n_threads_per_job=2` (4 threads total) is deliberately conservative --
another parallel job is already using ~20 of this machine's 32 cores. Re-tune
once that clears or if this machine is otherwise free.

In [3]:
cache_dir = dm.roots["mm"]/f"sub-{sub}"/"cst_cache"

cache_tasks = [
    ConnectomeCacheTask(label=f"{method}_pct{percentage}_rep{repeat}", args=(method, percentage, repeat))
    for method in methods
    for percentage in percentages[1:]
    for repeat in range(n_repeats)
]
print(f"{len(cache_tasks)} connectomes to cache: {[t.label for t in cache_tasks]}")

t0 = time.time()
cache_paths = cache_cst_batch(cache_tasks, corrupt, cache_dir, n_jobs=2, n_threads_per_job=2)
print(f"Stage 1 elapsed: {time.time()-t0:.1f}s")

75 connectomes to cache: ['swap_pct20_rep0', 'swap_pct20_rep1', 'swap_pct20_rep2', 'swap_pct20_rep3', 'swap_pct20_rep4', 'swap_pct40_rep0', 'swap_pct40_rep1', 'swap_pct40_rep2', 'swap_pct40_rep3', 'swap_pct40_rep4', 'swap_pct60_rep0', 'swap_pct60_rep1', 'swap_pct60_rep2', 'swap_pct60_rep3', 'swap_pct60_rep4', 'swap_pct80_rep0', 'swap_pct80_rep1', 'swap_pct80_rep2', 'swap_pct80_rep3', 'swap_pct80_rep4', 'swap_pct100_rep0', 'swap_pct100_rep1', 'swap_pct100_rep2', 'swap_pct100_rep3', 'swap_pct100_rep4', 'brutal_pct20_rep0', 'brutal_pct20_rep1', 'brutal_pct20_rep2', 'brutal_pct20_rep3', 'brutal_pct20_rep4', 'brutal_pct40_rep0', 'brutal_pct40_rep1', 'brutal_pct40_rep2', 'brutal_pct40_rep3', 'brutal_pct40_rep4', 'brutal_pct60_rep0', 'brutal_pct60_rep1', 'brutal_pct60_rep2', 'brutal_pct60_rep3', 'brutal_pct60_rep4', 'brutal_pct80_rep0', 'brutal_pct80_rep1', 'brutal_pct80_rep2', 'brutal_pct80_rep3', 'brutal_pct80_rep4', 'brutal_pct100_rep0', 'brutal_pct100_rep1', 'brutal_pct100_rep2', 'brutal_

caching connectome spectra: 100%|██████████| 72/72 [21:39<00:00, 18.05s/it]


Stage 1 elapsed: 1340.6s


## 🔧 - Stage 2 setup: runs + preprocess/reconstruct/postprocess

`preprocess_hrp_pp` picks hrp/pp from the *clean* (noise-free) `gold` projected
through the simulation forward model -- immune to noise, so hrp/pp indices don't
jitter across different runs/noise realizations (same fix `connectome_surrogates.ipynb`
already made over this notebook's original single-run version, now actually needed
here too since there are multiple runs/noise realizations in play).

`make_reconstruct` builds one `reconstruct_fn` per connectome (closes over that
connectome's `eigvec`/`eigval`), solving both `wCSET` and `CSET` per run.
`postprocess_score` scores both against that run's own `gold`, returning plain
picklable rows (no `MetricSuite.register` inside a worker -- that object only
exists in the main process).

In [4]:
runs = ["1", "2", "3"]     # small first pass -- see title cell

inverse_tasks = [
    InverseSimTask(
        run=int(r),
        simulator_dir=simulator_dir,
        fave=dm.get_path(sub, "ave", run=int(r)),
        fcov=dm.get_path(sub, "cov", run=int(r)),
        fgold=dm.get_path(sub, "gold", run=int(r)),
        fsource=dm.get_path(sub, "source", run=int(r)),
    )
    for r in runs
]


def preprocess_hrp_pp(ctx):
    ave, gold, simulator = ctx["ave"], ctx["gold"], ctx["simulator"]

    clean_ave = mne.apply_forward(simulator.hm.fwd, gold, ave.info)
    hrp_idx = mm.analysis.eeg.get_rising_phase(clean_ave, ratio=0.5, as_idx=True)[0]
    pp_idx = mm.analysis.eeg.get_rising_phase(clean_ave, ratio=1, as_idx=True)[0]

    phase_data = mne.EvokedArray(ave.data[:, [hrp_idx, pp_idx]], ave.info, tmin=0, nave=ave.nave, kind=ave.kind)
    ground_2pt = mm.analysis.inverse.make_stc(fwd["src"], gold.data[:, [hrp_idx, pp_idx]], tmin=0, tstep=1)

    return {"phase_data": phase_data, "ground_2pt": ground_2pt}


def make_reconstruct(fwd, eigvec, eigval):
    def reconstruct(ctx):
        phase_data, ncov = ctx["phase_data"], ctx["ncov"]

        # SNR from the real, full-time-reference `ave` -- not `phase_data`, whose
        # tmin=0 fake time axis would put the real baseline window out of range.
        snr, _ = mm.analysis.eeg.estimate_evoked_snr(evoked=ctx["ave"], baseline=(-0.4, -0.2), noise_cov=ncov)
        lambda2_wcset = max(0.01, 1.0 / (snr.max() ** 2))

        inv_wcset = mm.analysis.inverse.wCSET(
            forward=fwd, noise_cov=ncov, depth=0.5,
            eigenvectors=eigvec, eigenvalues=eigval,
            harm=5, harmonic_method="lowpass",
        )
        sol_wcset = inv_wcset.solve(phase_data, lambda2_wcset)

        inv_cset = mm.analysis.inverse.CSET(forward=fwd, noise_cov=ncov, depth=0.5, eigenvectors=eigvec)
        sol_cset = inv_cset.solve(phase_data, lambda2=0.1)

        return {"solutions": {"wcset": sol_wcset, "cset": sol_cset}}
    return reconstruct


def postprocess_score(ctx):
    ground_2pt = ctx["ground_2pt"]
    src, simulator = ctx["src"], ctx["simulator"]
    true_source_mask = src.select_targets(simulator.hm.src, simulator.hm.parc, simulator.hm.ras2tkras)

    rows = []
    for solver_name, sol in ctx["solutions"].items():
        corr = np.asarray(mm.analysis.inverse.compute_stc_correlation(ground_2pt, sol, mode="spatial", method="pearson"))
        pr_hrp = mm.core.calc_pr_curve(ground_2pt.data[:, 0] > 0, sol.data[:, 0])
        pr_pp = mm.core.calc_pr_curve(ground_2pt.data[:, 1] > 0, sol.data[:, 1])
        hit_hrp = find_peak_localization_error(sol.data[:, 0], true_source_mask, dipole_positions)

        rows.append(dict(
            run=ctx["task"].run, solver=solver_name,
            corr_hrp=float(corr[0]), corr_pp=float(corr[1]),
            ap_hrp=pr_hrp["ap"], ap_pp=pr_pp["ap"],
            loc_err_hrp=hit_hrp["distance_mm"],
        ))
    return rows

## 💥 - Stage 2: sequential connectomes x parallel runs

For each `(method, percentage, repeat)`: load its eigenbasis (clean, reused
directly, at `percentage=0`; else from Stage 1's cache), build a fresh
`reconstruct_fn` closed over that connectome, and call the *unmodified*
`run_parallel_pipeline` across all `runs`. Registered under
`label=f"{solver}_{method}"` (6 series total), `coord="percentage"`, same as
the original single-run version of this notebook -- just now with `n_repeats *
len(runs)` samples per point instead of `n_repeats`.

In [5]:
metrics = mm.metric.MetricSuite(f"Sabotage Connectome {sub} (multi-run)")
scalar_rows = []

t_start = time.time()

for method in methods:
    for percentage in tqdm(percentages, desc=f"{method} severity levels"):
        for repeat in range(n_repeats):
            if percentage == 0:
                eigvec, eigval = eigvec_clean, eigval_clean
            else:
                cached = np.load(cache_paths[f"{method}_pct{percentage}_rep{repeat}"])
                eigvec, eigval = cached["vec"], cached["val"]

            reconstruct_fn = make_reconstruct(fwd, eigvec, eigval)
            results = run_parallel_pipeline(
                inverse_tasks, preprocess_hrp_pp, reconstruct_fn, postprocess_score,
                n_jobs=len(runs), verbose="warning",
            )

            for run_rows in results:
                for row in run_rows:
                    label = f"{row['solver']}_{method}"
                    for metric_name, value in [
                        ("correlation_hrp", row["corr_hrp"]),
                        ("correlation_pp", row["corr_pp"]),
                        ("ap_hrp", row["ap_hrp"]),
                        ("ap_pp", row["ap_pp"]),
                        ("localization_error_hrp", row["loc_err_hrp"]),
                    ]:
                        metrics.register(label, metric_name, float(value), percentage=percentage)

                    scalar_rows.append(dict(
                        solver=row["solver"], method=method, percentage=percentage,
                        repeat=repeat, run=row["run"],
                        corr_hrp=row["corr_hrp"], corr_pp=row["corr_pp"],
                        ap_hrp=row["ap_hrp"], ap_pp=row["ap_pp"], loc_err_hrp=row["loc_err_hrp"],
                    ))

total_time = time.time() - t_start
scalar_df = pd.DataFrame(scalar_rows)
print(f"\nStage 2 total wall time: {total_time:.1f}s")
print(f"{len(scalar_df)} scored (solver, run) rows across {len(methods)} methods x {len(percentages)} percentages x {n_repeats} repeats x {len(runs)} runs")


swap severity levels:   0%|          | 0/2 [00:00<?, ?it/s]


pipeline:   0%|          | 0/3 [00:00<?, ?it/s]


pipeline:  33%|███▎      | 1/3 [00:03<00:06,  3.30s/it]


pipeline: 100%|██████████| 3/3 [00:03<00:00,  1.11s/it]


swap severity levels:  50%|█████     | 1/2 [00:34<00:34, 34.73s/it]


pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

... (16 outputs left out)


pipeline: 100%|██████████| 3/3 [00:00<00:00, 1035.80it/s]


dmri severity levels: 100%|██████████| 2/2 [00:55<00:00, 27.71s/it]


dmri severity levels: 100%|██████████| 2/2 [00:55<00:00, 27.81s/it]


Stage 2 total wall time: 166.1s
36 scored (solver, run) rows across 3 methods x 2 percentages x 1 repeats x 3 runs


## 📊 - Visualization

Correlation (hrp/pp), average precision (hrp/pp), and peak-localization-error (hrp) vs.
connectome corruption severity, one colored line per `{solver}_{method}` combination --
now aggregated (median/CI) across `n_repeats x len(runs)` samples per point instead of
`n_repeats` alone.

In [ ]:
import matplotlib.pyplot as plt

outcomes = ["correlation_hrp", "correlation_pp", "ap_hrp", "ap_pp", "localization_error_hrp"]
fig, axes = plt.subplots(1, len(outcomes), figsize=(5 * len(outcomes), 4))
for ax, outcome in zip(axes, outcomes):
    metrics.plot(metric=outcome, x="percentage", kind="line", ax=ax)
fig.tight_layout()
